# 04 — Second-Round Confirmatory Analysis

This notebook is the targeted extension added in response to the second-round review.

It does **not** replace the original four-dataset benchmark. It reads the frozen
dataset-local settings produced by Notebook 01, writes all new artifacts to
`outputs/second_round_confirmatory/`, and keeps the original benchmark outputs unchanged.

The workflow reproduces the additional analyses used in the second-round revision:

1. ten-seed focal confirmatory refits;
2. dependence-aware effect sizes and a revision-stage 1% relative-NMAE threshold;
3. validation-only reduced-architecture selection followed by a pre-test lock;
4. matched drop-one tests for components retained by the selected reduced model;
5. an explicit BDG external-validity scope audit; and
6. a common constrained storage-dispatch sensitivity experiment.

Generated result files remain local under `outputs/` and are not required to be
committed to GitHub.


In [ ]:
# 0. Repository paths and deterministic runtime settings.
# Execute this cell before importing torch.
from pathlib import Path
import os, sys, json, time, hashlib, platform, warnings, math, gc
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data"
ORIGINAL_RESULT_DIR = PROJECT_ROOT / "outputs"
SECOND_REVIEW_DIR = ORIGINAL_RESULT_DIR / "second_round_confirmatory"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

SECOND_REVIEW_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("PYTHONHASHSEED", "42")
for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(key, "1")

RUN_STARTED_UTC = datetime.now(timezone.utc)
RUN_STARTED_PERF = time.perf_counter()
PROTOCOL_NAME = "lash_second_review_confirmatory_20260829_v1"

OUTPUTS = {
    "protocol": SECOND_REVIEW_DIR / "00_protocol",
    "claim_audit": SECOND_REVIEW_DIR / "01_claim_and_external_scope",
    "seed_extension": SECOND_REVIEW_DIR / "02_ten_seed_confirmatory",
    "architecture": SECOND_REVIEW_DIR / "03_reduced_architecture",
    "effects": SECOND_REVIEW_DIR / "04_effect_sizes_and_inference",
    "scheduling": SECOND_REVIEW_DIR / "05_constrained_storage_scheduling",
    "response": SECOND_REVIEW_DIR / "06_manuscript_and_response_materials",
}
for p in OUTPUTS.values():
    p.mkdir(parents=True, exist_ok=True)

if not ORIGINAL_RESULT_DIR.exists():
    raise FileNotFoundError(
        "Run Notebook 01 first so the frozen benchmark outputs exist under outputs/."
    )

print("Frozen benchmark outputs:", ORIGINAL_RESULT_DIR)
print("Second-round outputs:", SECOND_REVIEW_DIR)
print("Protocol:", PROTOCOL_NAME)


In [ ]:
# 1. Verify the exact source modules used for the reported second-round run.
# Dependencies are installed from ../requirements.txt; install PyTorch separately
# using the CPU/CUDA build appropriate for the local machine.
import importlib.util

try:
    import torch
except ImportError as exc:
    raise RuntimeError(
        "PyTorch is missing. Install the appropriate PyTorch build, restart the kernel, and rerun."
    ) from exc

expected_module_sha256 = {
    "lash_deadline48.py": "8125a385ff40d60b49faa4fa83b4b6b5d7f33d89afd52fd65e264cfbd399b166",
    "lash_hardware_optimized.py": "71f60dd830a9ab6079653d9e5e44b8e7ad3edb4d1c0fbbf642de121091112f1a",
    "lash_per_dataset_hpo.py": "e116eb1fdc439cc4ea821e8aa06658205453892a3586329c1ef78f02647d1652",
    "lash_revision_ablation.py": "0908dac343661e19c97df0f05bd9ff0fd2cef1a297ced37ccaec82f0a45097b9",
    "lash_revision_analysis.py": "28591f97ffe7417546afcf6664c020c151c398747663d6b7659e119247a85633",
    "lash_revision_core.py": "ce918efe1b1598f0d2a810f4400979ba51ce94b11a6dbac0aae00e85039c088b"
}

for name, expected in expected_module_sha256.items():
    path = SRC_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Required source module missing: {path}")
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != expected:
        raise RuntimeError(
            f"Source hash mismatch for {name}. Expected {expected}, got {digest}. "
            "Use the repository version supplied with this revision."
        )

print("Source bundle verified.")
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)


In [ ]:
# 2. Hardware audit and portable execution policy.
import numpy as np
import pandas as pd
import psutil, cpuinfo
from IPython.display import display

import lash_revision_core as core
import lash_deadline48 as engine
import lash_per_dataset_hpo as local_hpo
import lash_revision_ablation as ablation
import lash_revision_analysis as analysis
from lash_revision_core import ExperimentConfig
from lash_per_dataset_hpo import PerDatasetHPOPolicy
from lash_hardware_optimized import apply_hardware_patch, probe_gpu_boosters

physical_cores = psutil.cpu_count(logical=False) or 1
logical_threads = psutil.cpu_count(logical=True) or physical_cores
ram_gb = psutil.virtual_memory().total / 1024**3
cpu_name = cpuinfo.get_cpu_info().get("brand_raw") or platform.processor() or "Unknown CPU"

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-only"
gpu_vram_gb = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
    if torch.cuda.is_available() else 0.0
)

# The reported run used Ryzen 7 7800X3D / 32 GB / RTX 5070 Ti.
# The code allows CPU fallback for independent reproduction, although it is slower.
TREE_HORIZON_JOBS = min(8, max(1, physical_cores))
TREE_THREADS_PER_MODEL = 1

torch.set_num_threads(min(8, max(1, logical_threads)))
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if hasattr(torch.backends.cuda.matmul, "allow_tf32"):
    torch.backends.cuda.matmul.allow_tf32 = False
if hasattr(torch.backends.cudnn, "allow_tf32"):
    torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True, warn_only=True)

apply_hardware_patch()
booster_probe = probe_gpu_boosters()

hardware = {
    "cpu": cpu_name,
    "physical_cores": physical_cores,
    "logical_threads": logical_threads,
    "ram_gb": round(ram_gb, 2),
    "gpu": gpu_name,
    "gpu_vram_gb": round(gpu_vram_gb, 2),
    "torch": torch.__version__,
    "cuda_build": str(torch.version.cuda),
    "tree_horizon_jobs": TREE_HORIZON_JOBS,
    "tree_threads_per_model": TREE_THREADS_PER_MODEL,
    "amp": bool(torch.cuda.is_available()),
    "cublas_workspace_config": os.environ.get("CUBLAS_WORKSPACE_CONFIG"),
    "tf32_disabled": True,
    **booster_probe,
}
display(pd.DataFrame([hardware]))
pd.DataFrame([hardware]).to_excel(OUTPUTS["protocol"] / "hardware_audit.xlsx", index=False)


In [ ]:
# 3. Verify the four harmonized inputs and the frozen primary run contract.
EXPECTED_CSVS = (
    "Cluster_1_Harmonized.csv",
    "Cluster_2_Harmonized.csv",
    "BDG_Edu_Harmonized.csv",
    "BDG_Dorm_Harmonized.csv",
)
ALL_DATASETS = ("CLUSTER_1", "CLUSTER_2", "BDG_EDU", "BDG_DORM")
CLUSTER_DATASETS = ("CLUSTER_1", "CLUSTER_2")
BDG_DATASETS = ("BDG_EDU", "BDG_DORM")

manifest_rows = []
for filename in EXPECTED_CSVS:
    path = DATA_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path)
    required = ["Date", "Holi", "Temp", "Humi", "WS", "Consumption"]
    if list(frame.columns) != required:
        raise ValueError(f"{filename}: expected {required}, got {list(frame.columns)}")
    manifest_rows.append({
        "file": filename,
        "source": str(path),
        "rows": len(frame),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    })
input_manifest = pd.DataFrame(manifest_rows)
input_manifest.to_csv(OUTPUTS["protocol"] / "input_manifest_second_review.csv", index=False)

original_contract_path = ORIGINAL_RESULT_DIR / "run_contract_20260823_per_dataset_hpo.json"
if not original_contract_path.exists():
    raise FileNotFoundError(
        f"{original_contract_path}\nRun Notebook 01 before the second-round extension."
    )
original_contract = json.loads(original_contract_path.read_text(encoding="utf-8"))
if original_contract.get("protocol") != "reviewer_20260823_per_dataset_hpo_v1":
    raise RuntimeError("Unexpected original protocol; incompatible experiments must not be mixed.")
if original_contract.get("test_used_for_selection") is True:
    raise RuntimeError("Original contract reports test-based selection; aborting.")

frozen_settings = {}
settings_snapshot_dir = OUTPUTS["protocol"] / "original_frozen_settings_snapshot"
settings_snapshot_dir.mkdir(parents=True, exist_ok=True)
for key in ALL_DATASETS:
    path = ORIGINAL_RESULT_DIR / "hpo_by_dataset" / key / "local_hpo_settings.json"
    if not path.exists():
        raise FileNotFoundError(path)
    payload = json.loads(path.read_text(encoding="utf-8"))
    if payload.get("status") != "complete" or payload.get("selection_dataset") != key:
        raise RuntimeError(f"{key}: local HPO settings are incomplete or mismatched")
    if payload.get("test_used_for_selection") is True:
        raise RuntimeError(f"{key}: test selection flag is not acceptable")
    frozen_settings[key] = payload
    (settings_snapshot_dir / f"{key}_local_hpo_settings.json").write_text(
        json.dumps(payload, indent=2), encoding="utf-8"
    )

print("Frozen primary contract verified.")
display(input_manifest)


In [ ]:
# 4. Freeze the second-round analysis plan BEFORE any new refit.
CONFIRMATORY_SEEDS = (42, 142, 242, 342, 442, 542, 642, 742, 842, 942)
HPO_SEEDS = (42, 142)
BOOTSTRAP_REPS = 5000
BLOCK_LENGTHS = (24, 72, 168, 336)
PRIMARY_BLOCK_HOURS = 168
SESOI_REL_NMAE_PCT = 1.0
ARCH_PARIMONY_TOL_PP = 0.010
ARCH_MAX_DATASET_PENALTY_PP = 0.030

# Nested reviewer-motivated reduced candidates. The sequence is frozen here;
# selection later uses only Cluster 1/2 validation data and never test loss.
ARCH_CANDIDATES = {
    "R1_NO_GATES_MEAN_POOL": {
        "ablation_flags": {"use_feature_gates": False, "learned_pooling": False},
        "include_nonlinear_weather": True,
        "include_phase_shift_calendar": True,
        "complexity_points": 5,
    },
    "R2_R1_NO_GRB": {
        "ablation_flags": {"use_feature_gates": False, "learned_pooling": False, "use_grb": False},
        "include_nonlinear_weather": True,
        "include_phase_shift_calendar": True,
        "complexity_points": 4,
    },
    "R3_R2_NO_HORIZON_EMBED": {
        "ablation_flags": {
            "use_feature_gates": False, "learned_pooling": False,
            "use_grb": False, "use_horizon_embedding": False,
        },
        "include_nonlinear_weather": True,
        "include_phase_shift_calendar": True,
        "complexity_points": 3,
    },
    "R4_R3_NO_PHASE_SHIFT": {
        "ablation_flags": {
            "use_feature_gates": False, "learned_pooling": False,
            "use_grb": False, "use_horizon_embedding": False,
        },
        "include_nonlinear_weather": True,
        "include_phase_shift_calendar": False,
        "complexity_points": 2,
    },
    "R5_R4_LINEAR_WEATHER_ONLY": {
        "ablation_flags": {
            "use_feature_gates": False, "learned_pooling": False,
            "use_grb": False, "use_horizon_embedding": False,
        },
        "include_nonlinear_weather": False,
        "include_phase_shift_calendar": False,
        "complexity_points": 1,
    },
}

# Storage size is normalized using PRETEST mean load, never test demand.
STORAGE_SCENARIOS = {
    "S": {"duration_h": 0.5, "power_fraction": 0.25},
    "M": {"duration_h": 1.0, "power_fraction": 0.50},
    "L": {"duration_h": 2.0, "power_fraction": 1.00},
}

second_review_plan = {
    "protocol": PROTOCOL_NAME,
    "frozen_utc": datetime.now(timezone.utc).isoformat(),
    "original_result_dir": str(ORIGINAL_RESULT_DIR),
    "second_review_dir": str(SECOND_REVIEW_DIR),
    "confirmatory_seeds": list(CONFIRMATORY_SEEDS),
    "hpo_seeds": list(HPO_SEEDS),
    "bootstrap_reps": BOOTSTRAP_REPS,
    "block_lengths_h": list(BLOCK_LENGTHS),
    "primary_block_h": PRIMARY_BLOCK_HOURS,
    "revision_stage_SESOI_relative_NMAE_pct": SESOI_REL_NMAE_PCT,
    "SESOI_scope": (
        "Fixed before the 20260829 additional refits as a reviewer-requested interpretation rule; "
        "not claimed as an original-study preregistration or universal BEMS threshold."
    ),
    "architecture_candidates": ARCH_CANDIDATES,
    "architecture_selection_rule": {
        "development_datasets": list(CLUSTER_DATASETS),
        "test_used_for_selection": False,
        "eligible_if_mean_calibration_score_within_pp": ARCH_PARIMONY_TOL_PP,
        "and_each_dataset_within_best_plus_pp": ARCH_MAX_DATASET_PENALTY_PP,
        "tie_break": "lowest complexity_points, then lowest mean validation-calibration score",
    },
    "storage_scenarios": STORAGE_SCENARIOS,
    "external_validity_claim": (
        "BDG_Edu and BDG_Dorm are external-source checks with dataset-local development; "
        "they are not cross-site, zero-shot, frozen-transfer, or cross-climate validation."
    ),
    "claim_tone": "competitive / supported where inferential evidence warrants; no universal superiority",
}
plan_path = OUTPUTS["protocol"] / "SECOND_REVIEW_PLAN_FROZEN_BEFORE_REFITS.json"
plan_path.write_text(json.dumps(second_review_plan, indent=2, ensure_ascii=False), encoding="utf-8")
plan_sha = hashlib.sha256(plan_path.read_bytes()).hexdigest()
(OUTPUTS["protocol"] / "SECOND_REVIEW_PLAN_SHA256.txt").write_text(plan_sha + "\n", encoding="utf-8")
print("Second-review plan frozen:", plan_path)
print("SHA-256:", plan_sha)

In [ ]:
# 5. Build second-review configurations.
policy = PerDatasetHPOPolicy(
    computational_budget_hours=48.0,
    bootstrap_reps=BOOTSTRAP_REPS,
    tree_final_seeds=CONFIRMATORY_SEEDS,
    neural_comparator_final_seeds=CONFIRMATORY_SEEDS,
    lash_final_seeds=CONFIRMATORY_SEEDS,
    mechanism_seeds=CONFIRMATORY_SEEDS,
)

config = ExperimentConfig(
    data_root=DATA_DIR,
    output_root=SECOND_REVIEW_DIR,
    dataset_keys=ALL_DATASETS,
    run_profile="paper",
    weather_mode="historical_only",
    hpo_seeds=HPO_SEEDS,
    final_refit_seeds=CONFIRMATORY_SEEDS,
    primary_hpo_repeats=2,
    external_hpo_repeats=2,
    hpo_trials_per_dimension=1,
    hpo_min_trials=3,
    hpo_max_trials=5,
    max_epochs=35,
    early_stopping_patience=5,
    tree_horizon_jobs=TREE_HORIZON_JOBS,
    tree_threads_per_model=TREE_THREADS_PER_MODEL,
    require_cuda=False,
    use_amp=True,
    save_models=False,
    resume=True,
)

print("Second-review config ready")
print("Final seeds:", config.final_refit_seeds)
print("Tree jobs:", config.tree_horizon_jobs, "x", config.tree_threads_per_model, "thread")

## Phase I — Reviewer comments 1 and 4: focal 10-seed confirmatory refits

The original HPO settings and router decisions are kept frozen. The only change is the number of final stochastic refits. This prevents the second-round seed extension from becoming a new test-driven tuning exercise.

The focal contrasts are fixed to those already named in the reviewer comment and manuscript: Cluster 1 vs LightGBM and Ridge, Cluster 2 vs Ridge, BDG_Edu vs Ridge, and BDG_Dorm vs Seasonal-24 and Ridge.

In [ ]:
# 6. Helpers for deterministic caching, full-LASH refits, and focal comparators.
from pathlib import Path
import joblib

FOCAL_COMPARATORS = {
    "CLUSTER_1": ("LIGHTGBM", "RIDGE"),
    "CLUSTER_2": ("RIDGE",),
    "BDG_EDU": ("RIDGE",),
    "BDG_DORM": ("SEASONAL_24", "RIDGE"),
}


def _prediction_file(root: Path, dataset_key: str, model: str, seed: int) -> Path:
    p = root / dataset_key / "predictions" / f"{model}__seed{seed}.npz"
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


def _runtime_file(root: Path, dataset_key: str, model: str, seed: int) -> Path:
    p = root / dataset_key / "runtime" / f"{model}__seed{seed}.json"
    p.parent.mkdir(parents=True, exist_ok=True)
    return p


def save_prediction(path: Path, bundle, prediction: np.ndarray, **extra):
    payload = {
        "forecast_origin": bundle.forecast_origin,
        "target_time": bundle.target_time,
        "actual": bundle.y.astype(np.float32),
        "prediction": np.asarray(prediction, np.float32),
    }
    for k, v in extra.items():
        payload[k] = np.asarray(v)
    np.savez_compressed(path, **payload)


def load_prediction(path: Path):
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}


def get_original_router_weights(dataset_key: str) -> np.ndarray:
    p = ORIGINAL_RESULT_DIR / "benchmark" / dataset_key / "configs" / "router_default.json"
    if not p.exists():
        raise FileNotFoundError(p)
    payload = json.loads(p.read_text(encoding="utf-8"))
    w = np.asarray(payload["weights"], dtype=float)
    if w.shape != (24,):
        raise ValueError(f"{dataset_key}: expected 24 router weights, got {w.shape}")
    return w


def build_full_bundles(dataset_key: str, *, nonlinear_weather=True, phase_shift=True):
    _, bundles = core.build_dataset_bundles(
        config,
        dataset_key,
        include_nonlinear_weather=nonlinear_weather,
        include_phase_shift_calendar=phase_shift,
    )
    return bundles


def fit_ridge_prediction(dataset_key: str, setting: dict, test_bundle=None, bundles=None):
    if bundles is None:
        bundles = build_full_bundles(dataset_key)
    train, val, test = bundles["train"], bundles["val"], bundles["test"]
    pretest = core.concat_bundles(train, val, split="train_plus_validation")
    fit = core.fit_ridge(pretest, float(setting["params"]["alpha"]))
    raw = core.predict_ridge_raw(fit, test)
    pred = core.apply_residual_gain(test, raw, float(setting["gain"]))
    return pred, fit, test


def run_full_lash_seed_extension(dataset_key: str):
    root = OUTPUTS["seed_extension"]
    settings = frozen_settings[dataset_key]["models"]
    bundles = build_full_bundles(dataset_key)
    train, val, test = bundles["train"], bundles["val"], bundles["test"]
    pretest = core.concat_bundles(train, val, split="train_plus_validation")

    # Ridge is deterministic and refitted once under the original frozen alpha/gain.
    ridge_setting = settings["RIDGE"]
    ridge_pred, ridge_fit, _ = fit_ridge_prediction(dataset_key, ridge_setting, bundles=bundles)
    ridge_path = _prediction_file(root, dataset_key, "RIDGE", 0)
    if not ridge_path.exists():
        save_prediction(ridge_path, test, ridge_pred)

    # Preserve the ORIGINAL validation-frozen router; do not reselect it from test.
    router_weights = get_original_router_weights(dataset_key)
    seq_setting = settings["LASH_SEQ"]

    rows = []
    for seed in CONFIRMATORY_SEEDS:
        lash_path = _prediction_file(root, dataset_key, "LASH", seed)
        seq_path = _prediction_file(root, dataset_key, "LASH_SEQ_COMPONENT", seed)
        runtime_path = _runtime_file(root, dataset_key, "LASH", seed)
        if lash_path.exists() and seq_path.exists() and runtime_path.exists():
            payload = load_prediction(lash_path)
            rows.append({
                "dataset": dataset_key, "model": "LASH", "seed": seed,
                **core.regression_metrics(payload["actual"], payload["prediction"]),
                **json.loads(runtime_path.read_text(encoding="utf-8")),
            })
            continue

        core.set_seed(seed)
        started = time.perf_counter()
        fit = core.fit_neural(
            "LASH_SEQ", pretest, seq_setting["params"], config,
            validation_bundle=None,
            fixed_epochs=int(seq_setting["best_epoch"]),
            seed=seed,
            ablation={},
        )
        seq_raw = core.predict_neural_raw(fit, test, seq_setting["params"], config)
        seq_pred = core.apply_residual_gain(test, seq_raw, float(seq_setting["gain"]))
        lash_pred = core.blend_predictions(seq_pred, ridge_pred, router_weights)
        elapsed = time.perf_counter() - started

        save_prediction(seq_path, test, seq_pred)
        save_prediction(lash_path, test, lash_pred)
        runtime = {
            "train_plus_inference_seconds": elapsed,
            "parameter_count_seq": int(fit.parameter_count),
            "router_mean_sequential_weight": float(np.mean(router_weights)),
            "router_min_sequential_weight": float(np.min(router_weights)),
            "router_max_sequential_weight": float(np.max(router_weights)),
            "source_setting": "original_20260823_frozen_local_HPO",
        }
        runtime_path.write_text(json.dumps(runtime, indent=2), encoding="utf-8")
        rows.append({
            "dataset": dataset_key, "model": "LASH", "seed": seed,
            **core.regression_metrics(test.y, lash_pred), **runtime,
        })
        del fit
        gc.collect(); torch.cuda.empty_cache()

    out = pd.DataFrame(rows)
    out.to_csv(root / dataset_key / "LASH_10seed_metrics.csv", index=False)
    return out


def run_focal_comparator(dataset_key: str, model: str):
    root = OUTPUTS["seed_extension"]
    settings = frozen_settings[dataset_key]["models"]
    bundles = build_full_bundles(dataset_key)
    train, val, test = bundles["train"], bundles["val"], bundles["test"]
    pretest = core.concat_bundles(train, val, split="train_plus_validation")

    if model == "RIDGE":
        path = _prediction_file(root, dataset_key, model, 0)
        if not path.exists():
            pred, _, _ = fit_ridge_prediction(dataset_key, settings["RIDGE"], bundles=bundles)
            save_prediction(path, test, pred)
        payload = load_prediction(path)
        return pd.DataFrame([{
            "dataset": dataset_key, "model": model, "seed": 0,
            **core.regression_metrics(payload["actual"], payload["prediction"]),
        }])

    if model == "SEASONAL_24":
        path = _prediction_file(root, dataset_key, model, 0)
        if not path.exists():
            save_prediction(path, test, test.naive24)
        payload = load_prediction(path)
        return pd.DataFrame([{
            "dataset": dataset_key, "model": model, "seed": 0,
            **core.regression_metrics(payload["actual"], payload["prediction"]),
        }])

    if model != "LIGHTGBM":
        raise ValueError(model)

    setting = settings["LIGHTGBM"]
    rows = []
    for seed in CONFIRMATORY_SEEDS:
        path = _prediction_file(root, dataset_key, model, seed)
        rt = _runtime_file(root, dataset_key, model, seed)
        if path.exists() and rt.exists():
            payload = load_prediction(path)
            rows.append({
                "dataset": dataset_key, "model": model, "seed": seed,
                **core.regression_metrics(payload["actual"], payload["prediction"]),
                **json.loads(rt.read_text(encoding="utf-8")),
            })
            continue
        core.set_seed(seed)
        started = time.perf_counter()
        fit = core.fit_direct_tree("LIGHTGBM", pretest, setting["params"], seed, config)
        raw = core.predict_direct_tree_raw(fit, test)
        pred = core.apply_residual_gain(test, raw, float(setting["gain"]))
        elapsed = time.perf_counter() - started
        save_prediction(path, test, pred)
        runtime = {"train_plus_inference_seconds": elapsed, "source_setting": "original_20260823_frozen_local_HPO"}
        rt.write_text(json.dumps(runtime, indent=2), encoding="utf-8")
        rows.append({
            "dataset": dataset_key, "model": model, "seed": seed,
            **core.regression_metrics(test.y, pred), **runtime,
        })
        del fit
        gc.collect()
    out = pd.DataFrame(rows)
    out.to_csv(root / dataset_key / f"{model}_10seed_metrics.csv", index=False)
    return out

In [ ]:
# 7. Execute the focal 10-seed extension.
seed_metric_parts = []
for dataset_key in ALL_DATASETS:
    print("=" * 84)
    print("10-SEED FULL LASH:", dataset_key)
    print("=" * 84)
    seed_metric_parts.append(run_full_lash_seed_extension(dataset_key))
    for comparator in FOCAL_COMPARATORS[dataset_key]:
        print("Comparator:", comparator)
        seed_metric_parts.append(run_focal_comparator(dataset_key, comparator))

seed_metrics = pd.concat(seed_metric_parts, ignore_index=True, sort=False)
seed_metrics.to_csv(OUTPUTS["seed_extension"] / "all_focal_seed_metrics.csv", index=False)

seed_summary = (
    seed_metrics.groupby(["dataset", "model"], as_index=False)
    .agg(
        n_seed=("seed", "count"),
        MAPE_mean=("MAPE", "mean"), MAPE_sd=("MAPE", "std"),
        CVRMSE_mean=("CVRMSE", "mean"), CVRMSE_sd=("CVRMSE", "std"),
        NMAE_mean=("NMAE", "mean"), NMAE_sd=("NMAE", "std"),
        composite_mean=("selection_score", "mean"), composite_sd=("selection_score", "std"),
    )
)
seed_summary.to_csv(OUTPUTS["seed_extension"] / "all_focal_seed_summary.csv", index=False)
display(seed_summary)

In [ ]:
# 8. Effect sizes + dependence-aware hierarchical inference + revision-stage practical threshold.
from statsmodels.stats.multitest import multipletests


def origin_nmae(payload):
    actual = np.asarray(payload["actual"], float)
    pred = np.asarray(payload["prediction"], float)
    denom = float(np.mean(actual))
    return np.abs(actual - pred).mean(axis=1) / denom * 100.0


def paired_seed_diffs(dataset_key: str, comparator: str):
    root = OUTPUTS["seed_extension"]
    diffs = {}
    comp_seeded = comparator == "LIGHTGBM"
    comparator_mean_losses = []
    lash_mean_losses = []
    for seed in CONFIRMATORY_SEEDS:
        lp = load_prediction(_prediction_file(root, dataset_key, "LASH", seed))
        cp_seed = seed if comp_seeded else 0
        cp = load_prediction(_prediction_file(root, dataset_key, comparator, cp_seed))
        l = origin_nmae(lp)
        c = origin_nmae(cp)
        if len(l) != len(c):
            raise ValueError(f"Origin mismatch: {dataset_key} {comparator} seed {seed}")
        diffs[seed] = l - c
        lash_mean_losses.append(l.mean())
        comparator_mean_losses.append(c.mean())
    return diffs, float(np.mean(lash_mean_losses)), float(np.mean(comparator_mean_losses))


def nonoverlap_block_effect(seed_diffs, block_length=168):
    stacked = np.stack([np.asarray(seed_diffs[s], float) for s in sorted(seed_diffs)])
    mean_diff = stacked.mean(axis=0)
    n_blocks = len(mean_diff) // block_length
    if n_blocks < 2:
        return {"n_blocks": n_blocks, "hedges_g_block": np.nan, "block_win_rate": np.nan}
    vals = mean_diff[: n_blocks * block_length].reshape(n_blocks, block_length).mean(axis=1)
    sd = vals.std(ddof=1)
    dz = 0.0 if sd == 0 else -vals.mean() / sd  # positive favors LASH
    correction = 1.0 - 3.0 / max(4.0 * n_blocks - 5.0, 1.0)
    return {
        "n_blocks": n_blocks,
        "hedges_g_block": float(dz * correction),
        "block_win_rate": float(np.mean(vals < 0)),
    }


rows = []
for dataset_key, comparators in FOCAL_COMPARATORS.items():
    for comparator in comparators:
        seed_diffs, lash_mean, comp_mean = paired_seed_diffs(dataset_key, comparator)
        for block in BLOCK_LENGTHS:
            boot = engine._batched_hierarchical_seed_block_bootstrap(
                seed_diffs,
                block_length=block,
                reps=BOOTSTRAP_REPS,
                seed=20260829 + block,
                batch_size=64,
            )
            mean_diff = float(np.mean([v.mean() for v in seed_diffs.values()]))
            relative_improvement = -100.0 * mean_diff / comp_mean
            rel_ci_low = -100.0 * boot["ci_high"] / comp_mean
            rel_ci_high = -100.0 * boot["ci_low"] / comp_mean
            block_fx = nonoverlap_block_effect(seed_diffs, block_length=block)
            rows.append({
                "dataset": dataset_key,
                "comparator": comparator,
                "block_h": block,
                "n_training_seeds": len(seed_diffs),
                "LASH_origin_NMAE_mean_pct": lash_mean,
                "comparator_origin_NMAE_mean_pct": comp_mean,
                "delta_NMAE_pp_LASH_minus_comparator": mean_diff,
                "relative_improvement_pct": relative_improvement,
                "relative_CI_low_pct": rel_ci_low,
                "relative_CI_high_pct": rel_ci_high,
                "CI_low_delta_pp": boot["ci_low"],
                "CI_high_delta_pp": boot["ci_high"],
                "p_hier_raw": boot["p_hier"],
                "SESOI_relative_NMAE_pct": SESOI_REL_NMAE_PCT,
                "point_meets_SESOI": bool(relative_improvement >= SESOI_REL_NMAE_PCT),
                "CI_supports_SESOI": bool(rel_ci_low >= SESOI_REL_NMAE_PCT),
                **block_fx,
            })

effect_table = pd.DataFrame(rows)

# Holm correction within each block length over the fixed focal contrast family.
effect_table["p_hier_holm"] = np.nan
for block, idx in effect_table.groupby("block_h").groups.items():
    p = effect_table.loc[idx, "p_hier_raw"].to_numpy(float)
    effect_table.loc[idx, "p_hier_holm"] = multipletests(p, method="holm")[1]

effect_table["statistically_supported_superiority"] = (
    (effect_table["p_hier_holm"] < 0.05)
    & (effect_table["CI_high_delta_pp"] < 0)
)

def interpretation(row):
    if row["CI_supports_SESOI"]:
        return "statistically/practically supported at revision-stage SESOI"
    if row["statistically_supported_superiority"] and row["point_meets_SESOI"]:
        return "statistically supported; practical threshold met by point estimate but not CI"
    if row["statistically_supported_superiority"]:
        return "statistically supported but below revision-stage practical threshold"
    if row["delta_NMAE_pp_LASH_minus_comparator"] < 0:
        return "numerically favorable; superiority not supported"
    if abs(row["delta_NMAE_pp_LASH_minus_comparator"]) < 1e-12:
        return "identical"
    return "comparator numerically favorable; no LASH superiority claim"

effect_table["interpretation"] = effect_table.apply(interpretation, axis=1)
effect_table.to_csv(OUTPUTS["effects"] / "focal_effect_sizes_and_hierarchical_inference.csv", index=False)

primary_effects = effect_table.loc[effect_table["block_h"].eq(PRIMARY_BLOCK_HOURS)].copy()
primary_effects.to_csv(OUTPUTS["effects"] / "PRIMARY_168h_effect_table.csv", index=False)
display(primary_effects)

## Phase II — Reviewer comment 3: validation-selected reduced architecture

The reduced architecture is not selected from Table 11 test losses. The nested candidate set was frozen in the protocol cell above. Candidate HPO uses only each dataset's `Train` and `Val_tune` periods; candidate routing is calibrated on `Val_calibration`. The **common architecture is selected using Cluster 1 and Cluster 2 validation-calibration scores only**, written to a lock file, and only then is any selected-reduced-model test evaluation executed.

This is intentionally more conservative than simply choosing whichever ablation happened to look best on the existing test table.

In [ ]:
# 9. Validation-only architecture development helpers. No test split is constructed here.

def build_train_val_only(dataset_key: str, *, nonlinear_weather=True, phase_shift=True):
    spec = config.specs[dataset_key]
    base = core.load_base_frame(spec, DATA_DIR)
    frame = core.add_causal_features(base, yoy_components=())
    past_cols, future_cols = core.feature_contract(
        yoy_components=(),
        include_nonlinear_weather=nonlinear_weather,
        include_phase_shift_calendar=phase_shift,
    )
    train = core.build_windows(frame, spec, "train", past_cols, future_cols)
    val = core.build_windows(frame, spec, "val", past_cols, future_cols)
    return train, val


def candidate_dir(dataset_key: str, candidate_name: str) -> Path:
    d = OUTPUTS["architecture"] / "development" / dataset_key / candidate_name
    for sub in ("tables", "calibration", "configs"):
        (d / sub).mkdir(parents=True, exist_ok=True)
    return d


def develop_reduced_candidate(dataset_key: str, candidate_name: str, spec: dict):
    d = candidate_dir(dataset_key, candidate_name)
    frozen_path = d / "configs" / "validation_only_settings.json"
    summary_path = d / "tables" / "validation_calibration_summary.csv"
    if frozen_path.exists() and summary_path.exists():
        return json.loads(frozen_path.read_text(encoding="utf-8")), pd.read_csv(summary_path)

    train, val = build_train_val_only(
        dataset_key,
        nonlinear_weather=spec["include_nonlinear_weather"],
        phase_shift=spec["include_phase_shift_calendar"],
    )
    val_tune, val_cal = core.split_validation_bundle(val)
    pre_cal = core.concat_bundles(train, val_tune, split="train_plus_val_tune")

    # Equal local selection contract for each candidate.
    ridge_sel, ridge_search = core.tune_ridge(train, val_tune)
    seq_sel, seq_trials, seq_repeats = core.tune_neural_repeated(
        config,
        dataset_key,
        "LASH_SEQ",
        train,
        val_tune,
        ablation=spec["ablation_flags"],
        study_suffix="__secondreview__" + candidate_name.lower(),
    )

    cal_seq_fit = core.fit_neural(
        "LASH_SEQ", pre_cal, seq_sel["params"], config,
        validation_bundle=None,
        fixed_epochs=int(seq_sel["best_epoch"]),
        seed=42,
        ablation=spec["ablation_flags"],
    )
    cal_seq_raw = core.predict_neural_raw(cal_seq_fit, val_cal, seq_sel["params"], config)
    cal_seq = core.apply_residual_gain(val_cal, cal_seq_raw, float(seq_sel["gain"]))

    cal_ridge_fit = core.fit_ridge(pre_cal, float(ridge_sel["alpha"]))
    cal_ridge_raw = core.predict_ridge_raw(cal_ridge_fit, val_cal)
    cal_ridge = core.apply_residual_gain(val_cal, cal_ridge_raw, float(ridge_sel["gain"]))

    router = core.select_router(val_cal, cal_seq, cal_ridge)
    routed = core.blend_predictions(cal_seq, cal_ridge, router.sequential_weights)

    summary = pd.DataFrame([
        {"dataset": dataset_key, "candidate": candidate_name, "component": "SEQUENTIAL", **core.regression_metrics(val_cal.y, cal_seq)},
        {"dataset": dataset_key, "candidate": candidate_name, "component": "RIDGE", **core.regression_metrics(val_cal.y, cal_ridge)},
        {"dataset": dataset_key, "candidate": candidate_name, "component": "ROUTED", **core.regression_metrics(val_cal.y, routed)},
    ])
    summary["complexity_points"] = int(spec["complexity_points"])
    summary.to_csv(summary_path, index=False)
    ridge_search.to_csv(d / "tables" / "ridge_search.csv", index=False)
    seq_trials.to_csv(d / "tables" / "sequential_hpo_trials.csv", index=False)
    seq_repeats.to_csv(d / "tables" / "sequential_hpo_repeats.csv", index=False)
    router.table.to_csv(d / "tables" / "router_candidates.csv", index=False)
    np.savez_compressed(
        d / "calibration" / "experts.npz",
        actual=val_cal.y.astype(np.float32),
        sequential=cal_seq.astype(np.float32),
        ridge=cal_ridge.astype(np.float32),
        routed=routed.astype(np.float32),
        forecast_origin=val_cal.forecast_origin,
        target_time=val_cal.target_time,
    )

    settings = {
        "dataset": dataset_key,
        "candidate": candidate_name,
        "selection_scope": "train + val_tune HPO; val_calibration router only; test not constructed",
        "test_used_for_selection": False,
        "ridge": {"alpha": float(ridge_sel["alpha"]), "gain": float(ridge_sel["gain"])},
        "sequential": {
            "params": seq_sel["params"],
            "best_epoch": int(seq_sel["best_epoch"]),
            "gain": float(seq_sel["gain"]),
            "selected_hpo_seed": int(seq_sel["selected_hpo_seed"]),
        },
        "router": {
            "mode": router.mode,
            "weights": router.sequential_weights.tolist(),
            "calibration_score": float(router.calibration_score),
            "best_single_score": float(router.best_single_score),
        },
        "candidate_spec": spec,
    }
    frozen_path.write_text(json.dumps(settings, indent=2), encoding="utf-8")

    del cal_seq_fit, cal_ridge_fit
    gc.collect(); torch.cuda.empty_cache()
    return settings, summary

In [ ]:
# 10. Develop all nested reduced candidates on Cluster 1 and Cluster 2; lock common architecture BEFORE test.
architecture_rows = []
architecture_settings = {}
for dataset_key in CLUSTER_DATASETS:
    architecture_settings[dataset_key] = {}
    for candidate_name, spec in ARCH_CANDIDATES.items():
        print("DEVELOP", dataset_key, candidate_name)
        settings, summary = develop_reduced_candidate(dataset_key, candidate_name, spec)
        architecture_settings[dataset_key][candidate_name] = settings
        routed = summary.loc[summary["component"].eq("ROUTED")].iloc[0]
        architecture_rows.append({
            "dataset": dataset_key,
            "candidate": candidate_name,
            "validation_calibration_score": float(routed["selection_score"]),
            "MAPE": float(routed["MAPE"]),
            "CVRMSE": float(routed["CVRMSE"]),
            "NMAE": float(routed["NMAE"]),
            "complexity_points": int(ARCH_CANDIDATES[candidate_name]["complexity_points"]),
            "router_mean_seq_weight": float(np.mean(settings["router"]["weights"])),
        })

arch_dev = pd.DataFrame(architecture_rows)
arch_dev.to_csv(OUTPUTS["architecture"] / "cluster_validation_only_candidate_scores.csv", index=False)

best_by_dataset = arch_dev.groupby("dataset")["validation_calibration_score"].min().to_dict()
agg_rows = []
for candidate_name, group in arch_dev.groupby("candidate"):
    mean_score = float(group["validation_calibration_score"].mean())
    max_penalty = float(max(
        row.validation_calibration_score - best_by_dataset[row.dataset]
        for row in group.itertuples()
    ))
    agg_rows.append({
        "candidate": candidate_name,
        "mean_validation_calibration_score": mean_score,
        "max_dataset_penalty_pp": max_penalty,
        "complexity_points": int(group["complexity_points"].iloc[0]),
    })
arch_agg = pd.DataFrame(agg_rows)
best_mean = float(arch_agg["mean_validation_calibration_score"].min())
arch_agg["within_mean_tolerance"] = arch_agg["mean_validation_calibration_score"] <= best_mean + ARCH_PARIMONY_TOL_PP
arch_agg["within_each_dataset_tolerance"] = arch_agg["max_dataset_penalty_pp"] <= ARCH_MAX_DATASET_PENALTY_PP
arch_agg["eligible_for_parsimony_selection"] = arch_agg["within_mean_tolerance"] & arch_agg["within_each_dataset_tolerance"]

eligible = arch_agg.loc[arch_agg["eligible_for_parsimony_selection"]].copy()
if eligible.empty:
    eligible = arch_agg.nsmallest(1, "mean_validation_calibration_score").copy()
    selection_note = "No candidate met both tolerance rules; selected best mean validation-calibration score."
else:
    selection_note = "Selected most parsimonious candidate among validation-eligible candidates."

selected_row = eligible.sort_values(
    ["complexity_points", "mean_validation_calibration_score"],
    ascending=[True, True],
).iloc[0]
SELECTED_REDUCED_ARCH = str(selected_row["candidate"])
SELECTED_REDUCED_SPEC = ARCH_CANDIDATES[SELECTED_REDUCED_ARCH]

arch_agg.to_csv(OUTPUTS["architecture"] / "architecture_selection_table.csv", index=False)
selection_lock = {
    "selected_architecture": SELECTED_REDUCED_ARCH,
    "selected_spec": SELECTED_REDUCED_SPEC,
    "selection_note": selection_note,
    "selection_datasets": list(CLUSTER_DATASETS),
    "selection_data": "validation only",
    "test_used_for_selection": False,
    "selected_utc": datetime.now(timezone.utc).isoformat(),
    "plan_sha256": plan_sha,
}
lock_path = OUTPUTS["architecture"] / "SELECTED_REDUCED_ARCHITECTURE_LOCKED_BEFORE_TEST.json"
lock_path.write_text(json.dumps(selection_lock, indent=2), encoding="utf-8")
lock_sha = hashlib.sha256(lock_path.read_bytes()).hexdigest()
(OUTPUTS["architecture"] / "SELECTED_REDUCED_ARCHITECTURE_SHA256.txt").write_text(lock_sha + "\n", encoding="utf-8")

print("LOCKED reduced architecture:", SELECTED_REDUCED_ARCH)
print("Lock SHA-256:", lock_sha)
display(arch_dev.sort_values(["dataset", "validation_calibration_score"]))
display(arch_agg.sort_values(["eligible_for_parsimony_selection", "complexity_points", "mean_validation_calibration_score"], ascending=[False, True, True]))

In [ ]:
# 11. After architecture lock: develop the selected reduced architecture on BDG using local train/validation only.
for dataset_key in BDG_DATASETS:
    print("BDG LOCAL DEVELOPMENT AFTER ARCHITECTURE LOCK:", dataset_key, SELECTED_REDUCED_ARCH)
    settings, summary = develop_reduced_candidate(dataset_key, SELECTED_REDUCED_ARCH, SELECTED_REDUCED_SPEC)
    architecture_settings.setdefault(dataset_key, {})[SELECTED_REDUCED_ARCH] = settings

# Reload cluster selected settings from disk to make the boundary explicit.
for dataset_key in CLUSTER_DATASETS:
    p = candidate_dir(dataset_key, SELECTED_REDUCED_ARCH) / "configs" / "validation_only_settings.json"
    architecture_settings.setdefault(dataset_key, {})[SELECTED_REDUCED_ARCH] = json.loads(p.read_text(encoding="utf-8"))

In [ ]:
# 12. Final 10-seed test evaluation of the validation-selected reduced architecture.

def reduced_test_dir(dataset_key: str) -> Path:
    d = OUTPUTS["architecture"] / "selected_reduced_test" / dataset_key
    for sub in ("predictions", "runtime", "tables"):
        (d / sub).mkdir(parents=True, exist_ok=True)
    return d


def run_selected_reduced_test(dataset_key: str):
    d = reduced_test_dir(dataset_key)
    setting = architecture_settings[dataset_key][SELECTED_REDUCED_ARCH]
    spec = SELECTED_REDUCED_SPEC
    _, bundles = core.build_dataset_bundles(
        config,
        dataset_key,
        include_nonlinear_weather=spec["include_nonlinear_weather"],
        include_phase_shift_calendar=spec["include_phase_shift_calendar"],
    )
    train, val, test = bundles["train"], bundles["val"], bundles["test"]
    pretest = core.concat_bundles(train, val, split="train_plus_validation")

    ridge = setting["ridge"]
    ridge_fit = core.fit_ridge(pretest, float(ridge["alpha"]))
    ridge_raw = core.predict_ridge_raw(ridge_fit, test)
    ridge_pred = core.apply_residual_gain(test, ridge_raw, float(ridge["gain"]))
    save_prediction(d / "predictions" / "RIDGE__seed0.npz", test, ridge_pred)

    seq_setting = setting["sequential"]
    weights = np.asarray(setting["router"]["weights"], float)
    rows = []
    for seed in CONFIRMATORY_SEEDS:
        routed_path = d / "predictions" / f"LASH_REDUCED__seed{seed}.npz"
        seq_path = d / "predictions" / f"LASH_REDUCED_SEQ__seed{seed}.npz"
        rt_path = d / "runtime" / f"seed{seed}.json"
        if routed_path.exists() and seq_path.exists() and rt_path.exists():
            payload = load_prediction(routed_path)
            rows.append({
                "dataset": dataset_key, "model": "LASH_REDUCED", "seed": seed,
                **core.regression_metrics(payload["actual"], payload["prediction"]),
                **json.loads(rt_path.read_text(encoding="utf-8")),
            })
            continue
        core.set_seed(seed)
        started = time.perf_counter()
        fit = core.fit_neural(
            "LASH_SEQ", pretest, seq_setting["params"], config,
            validation_bundle=None,
            fixed_epochs=int(seq_setting["best_epoch"]),
            seed=seed,
            ablation=spec["ablation_flags"],
        )
        seq_raw = core.predict_neural_raw(fit, test, seq_setting["params"], config)
        seq_pred = core.apply_residual_gain(test, seq_raw, float(seq_setting["gain"]))
        routed = core.blend_predictions(seq_pred, ridge_pred, weights)
        elapsed = time.perf_counter() - started
        save_prediction(seq_path, test, seq_pred)
        save_prediction(routed_path, test, routed)
        runtime = {
            "train_plus_inference_seconds": elapsed,
            "parameter_count_seq": int(fit.parameter_count),
            "router_mean_sequential_weight": float(np.mean(weights)),
        }
        rt_path.write_text(json.dumps(runtime, indent=2), encoding="utf-8")
        rows.append({
            "dataset": dataset_key, "model": "LASH_REDUCED", "seed": seed,
            **core.regression_metrics(test.y, routed), **runtime,
        })
        del fit
        gc.collect(); torch.cuda.empty_cache()
    out = pd.DataFrame(rows)
    out.to_csv(d / "tables" / "ten_seed_metrics.csv", index=False)
    return out

reduced_metrics = pd.concat([run_selected_reduced_test(k) for k in ALL_DATASETS], ignore_index=True)
reduced_metrics.to_csv(OUTPUTS["architecture"] / "selected_reduced_10seed_metrics_all_datasets.csv", index=False)
display(reduced_metrics.groupby("dataset")[["MAPE", "CVRMSE", "NMAE", "selection_score"]].agg(["mean", "std"]))

In [ ]:
# 13. Drop-one tests for EVERY optional component retained by the selected reduced architecture.
# Primary estimand = sequential-only change before router compensation.

def active_optional_components(spec):
    flags = {
        "FEATURE_GATES": spec["ablation_flags"].get("use_feature_gates", True),
        "LEARNED_POOLING": spec["ablation_flags"].get("learned_pooling", True),
        "CAUSAL_TCN": spec["ablation_flags"].get("use_tcn", True),
        "GRB": spec["ablation_flags"].get("use_grb", True),
        "HORIZON_EMBEDDING": spec["ablation_flags"].get("use_horizon_embedding", True),
        "NONLINEAR_WEATHER": spec["include_nonlinear_weather"],
        "PHASE_SHIFT_CALENDAR": spec["include_phase_shift_calendar"],
    }
    return [k for k, v in flags.items() if v]


def remove_component(spec, component):
    new = json.loads(json.dumps(spec))
    if component == "FEATURE_GATES": new["ablation_flags"]["use_feature_gates"] = False
    elif component == "LEARNED_POOLING": new["ablation_flags"]["learned_pooling"] = False
    elif component == "CAUSAL_TCN": new["ablation_flags"]["use_tcn"] = False
    elif component == "GRB": new["ablation_flags"]["use_grb"] = False
    elif component == "HORIZON_EMBEDDING": new["ablation_flags"]["use_horizon_embedding"] = False
    elif component == "NONLINEAR_WEATHER": new["include_nonlinear_weather"] = False
    elif component == "PHASE_SHIFT_CALENDAR": new["include_phase_shift_calendar"] = False
    else: raise ValueError(component)
    return new


def run_reduced_drop_one(dataset_key: str, component: str):
    base_setting = architecture_settings[dataset_key][SELECTED_REDUCED_ARCH]
    variant_spec = remove_component(SELECTED_REDUCED_SPEC, component)
    d = OUTPUTS["architecture"] / "retained_component_drop_one" / dataset_key / f"MINUS_{component}"
    for sub in ("predictions", "tables", "configs"):
        (d / sub).mkdir(parents=True, exist_ok=True)
    summary_path = d / "tables" / "metrics_by_seed.csv"

    _, bundles = core.build_dataset_bundles(
        config,
        dataset_key,
        include_nonlinear_weather=variant_spec["include_nonlinear_weather"],
        include_phase_shift_calendar=variant_spec["include_phase_shift_calendar"],
    )
    train, val, test = bundles["train"], bundles["val"], bundles["test"]
    val_tune, val_cal = core.split_validation_bundle(val)
    pre_cal = core.concat_bundles(train, val_tune, split="train_plus_val_tune")
    pretest = core.concat_bundles(train, val, split="train_plus_validation")

    # Hold reduced-model settings fixed to isolate the component.
    ridge = base_setting["ridge"]
    seq = base_setting["sequential"]

    cal_ridge_fit = core.fit_ridge(pre_cal, float(ridge["alpha"]))
    cal_ridge = core.apply_residual_gain(
        val_cal, core.predict_ridge_raw(cal_ridge_fit, val_cal), float(ridge["gain"])
    )
    cal_seq_fit = core.fit_neural(
        "LASH_SEQ", pre_cal, seq["params"], config,
        validation_bundle=None, fixed_epochs=int(seq["best_epoch"]), seed=42,
        ablation=variant_spec["ablation_flags"],
    )
    cal_seq = core.apply_residual_gain(
        val_cal, core.predict_neural_raw(cal_seq_fit, val_cal, seq["params"], config), float(seq["gain"])
    )
    router = core.select_router(val_cal, cal_seq, cal_ridge)

    final_ridge = core.fit_ridge(pretest, float(ridge["alpha"]))
    ridge_test = core.apply_residual_gain(
        test, core.predict_ridge_raw(final_ridge, test), float(ridge["gain"])
    )

    rows = []
    if summary_path.exists():
        prior = pd.read_csv(summary_path)
        rows.extend(prior.to_dict(orient="records"))
    completed = {(str(r["estimand"]), int(r["seed"])) for r in rows}

    for seed in CONFIRMATORY_SEEDS:
        seq_cache = d / "predictions" / f"SEQUENTIAL_ONLY_PRIMARY__seed{seed}.npz"
        routed_cache = d / "predictions" / f"REROUTED_SECONDARY__seed{seed}.npz"
        if (("SEQUENTIAL_ONLY_PRIMARY", seed) in completed
                and ("REROUTED_SECONDARY", seed) in completed
                and seq_cache.exists() and routed_cache.exists()):
            continue

        core.set_seed(seed)
        fit = core.fit_neural(
            "LASH_SEQ", pretest, seq["params"], config,
            validation_bundle=None, fixed_epochs=int(seq["best_epoch"]), seed=seed,
            ablation=variant_spec["ablation_flags"],
        )
        seq_test = core.apply_residual_gain(
            test, core.predict_neural_raw(fit, test, seq["params"], config), float(seq["gain"])
        )
        routed = core.blend_predictions(seq_test, ridge_test, router.sequential_weights)
        for estimand, pred, cache_path in (
            ("SEQUENTIAL_ONLY_PRIMARY", seq_test, seq_cache),
            ("REROUTED_SECONDARY", routed, routed_cache),
        ):
            rows = [r for r in rows if not (str(r.get("estimand")) == estimand and int(r.get("seed", -1)) == seed)]
            rows.append({
                "dataset": dataset_key,
                "component_removed": component,
                "estimand": estimand,
                "seed": seed,
                **core.regression_metrics(test.y, pred),
            })
            save_prediction(cache_path, test, pred)
        pd.DataFrame(rows).sort_values(["estimand", "seed"]).to_csv(summary_path, index=False)
        del fit
        gc.collect(); torch.cuda.empty_cache()

    pd.DataFrame(rows).sort_values(["estimand", "seed"]).to_csv(summary_path, index=False)
    router.table.to_csv(d / "tables" / "router_calibration.csv", index=False)
    (d / "configs" / "drop_one_protocol.json").write_text(json.dumps({
        "base_architecture": SELECTED_REDUCED_ARCH,
        "removed_component": component,
        "base_settings_frozen": True,
        "variant_spec": variant_spec,
        "primary_estimand": "SEQUENTIAL_ONLY_PRIMARY",
        "secondary_estimand": "REROUTED_SECONDARY",
        "test_used_for_selection": False,
    }, indent=2), encoding="utf-8")
    return pd.DataFrame(rows)

retained_components = active_optional_components(SELECTED_REDUCED_SPEC)
print("Retained optional components requiring direct evidence:", retained_components)

drop_tables = []
for dataset_key in ALL_DATASETS:
    for component in retained_components:
        print("DROP-ONE", dataset_key, component)
        drop_tables.append(run_reduced_drop_one(dataset_key, component))

drop_metrics = pd.concat(drop_tables, ignore_index=True) if drop_tables else pd.DataFrame()
drop_metrics.to_csv(OUTPUTS["architecture"] / "retained_component_drop_one_metrics.csv", index=False)

In [ ]:
# 14. Full vs reduced inference + retained-component incremental-value table.

def load_reduced_pred(dataset_key, seed, seq=False):
    name = "LASH_REDUCED_SEQ" if seq else "LASH_REDUCED"
    return load_prediction(reduced_test_dir(dataset_key) / "predictions" / f"{name}__seed{seed}.npz")


def load_full_pred(dataset_key, seed, seq=False):
    model = "LASH_SEQ_COMPONENT" if seq else "LASH"
    return load_prediction(_prediction_file(OUTPUTS["seed_extension"], dataset_key, model, seed))

# A. Reduced vs full routed pipeline.
reduced_vs_full_rows = []
for dataset_key in ALL_DATASETS:
    seed_diffs = {}
    full_mean = []
    reduced_mean = []
    for seed in CONFIRMATORY_SEEDS:
        r = origin_nmae(load_reduced_pred(dataset_key, seed))
        f = origin_nmae(load_full_pred(dataset_key, seed))
        seed_diffs[seed] = r - f  # positive = reduced worse
        full_mean.append(f.mean()); reduced_mean.append(r.mean())
    boot = engine._batched_hierarchical_seed_block_bootstrap(
        seed_diffs, block_length=PRIMARY_BLOCK_HOURS,
        reps=BOOTSTRAP_REPS, seed=20260829, batch_size=64,
    )
    fmean = float(np.mean(full_mean)); rmean = float(np.mean(reduced_mean))
    rel_degradation = 100.0 * (rmean - fmean) / fmean
    rel_ci_low = 100.0 * boot["ci_low"] / fmean
    rel_ci_high = 100.0 * boot["ci_high"] / fmean

    full_runtime = pd.read_csv(OUTPUTS["seed_extension"] / dataset_key / "LASH_10seed_metrics.csv")
    reduced_runtime = pd.read_csv(reduced_test_dir(dataset_key) / "tables" / "ten_seed_metrics.csv")
    full_params = float(full_runtime["parameter_count_seq"].dropna().mean())
    reduced_params = float(reduced_runtime["parameter_count_seq"].dropna().mean())
    full_seconds = float(full_runtime["train_plus_inference_seconds"].dropna().mean())
    reduced_seconds = float(reduced_runtime["train_plus_inference_seconds"].dropna().mean())

    reduced_vs_full_rows.append({
        "dataset": dataset_key,
        "reduced_architecture": SELECTED_REDUCED_ARCH,
        "full_origin_NMAE_mean_pct": fmean,
        "reduced_origin_NMAE_mean_pct": rmean,
        "delta_reduced_minus_full_pp": rmean - fmean,
        "relative_degradation_pct": rel_degradation,
        "relative_CI_low_pct": rel_ci_low,
        "relative_CI_high_pct": rel_ci_high,
        "noninferiority_margin_relative_pct": SESOI_REL_NMAE_PCT,
        "noninferior_within_revision_margin": bool(rel_ci_high < SESOI_REL_NMAE_PCT),
        "p_hier_two_sided": boot["p_hier"],
        "full_seq_parameter_count": full_params,
        "reduced_seq_parameter_count": reduced_params,
        "parameter_reduction_pct": 100.0 * (full_params - reduced_params) / full_params if full_params > 0 else np.nan,
        "full_train_plus_inference_seconds_mean": full_seconds,
        "reduced_train_plus_inference_seconds_mean": reduced_seconds,
        "runtime_reduction_pct": 100.0 * (full_seconds - reduced_seconds) / full_seconds if full_seconds > 0 else np.nan,
    })
reduced_vs_full = pd.DataFrame(reduced_vs_full_rows)
reduced_vs_full.to_csv(OUTPUTS["effects"] / "reduced_vs_full_noninferiority.csv", index=False)

# B. Every retained optional component: primary sequential-only drop-one contrast.
component_rows = []
if retained_components:
    for dataset_key in ALL_DATASETS:
        for component in retained_components:
            seed_diffs = {}
            for seed in CONFIRMATORY_SEEDS:
                base = origin_nmae(load_reduced_pred(dataset_key, seed, seq=True))
                d = OUTPUTS["architecture"] / "retained_component_drop_one" / dataset_key / f"MINUS_{component}"
                minus = origin_nmae(load_prediction(d / "predictions" / f"SEQUENTIAL_ONLY_PRIMARY__seed{seed}.npz"))
                seed_diffs[seed] = minus - base  # positive => removal hurts => component adds value
            boot = engine._batched_hierarchical_seed_block_bootstrap(
                seed_diffs, block_length=PRIMARY_BLOCK_HOURS,
                reps=BOOTSTRAP_REPS, seed=20260829 + len(component), batch_size=64,
            )
            mean_delta = float(np.mean([v.mean() for v in seed_diffs.values()]))
            component_rows.append({
                "dataset": dataset_key,
                "retained_component": component,
                "delta_NMAE_pp_removal_minus_base": mean_delta,
                "CI_low": boot["ci_low"],
                "CI_high": boot["ci_high"],
                "p_raw": boot["p_hier"],
                "direction": "supports_retention" if mean_delta > 0 else "no_incremental_gain_detected",
            })
component_effects = pd.DataFrame(component_rows)
if not component_effects.empty:
    component_effects["p_holm"] = np.nan
    for dataset_key, idx in component_effects.groupby("dataset").groups.items():
        component_effects.loc[idx, "p_holm"] = multipletests(
            component_effects.loc[idx, "p_raw"].to_numpy(float), method="holm"
        )[1]
    component_effects["supported_after_Holm"] = (
        (component_effects["p_holm"] < 0.05)
        & (component_effects["CI_low"] > 0)
    )
    component_effects.to_csv(OUTPUTS["effects"] / "retained_component_incremental_value.csv", index=False)

print("Reduced vs full:")
display(reduced_vs_full)
if not component_effects.empty:
    print("Retained-component evidence:")
    display(component_effects)

## Phase III — Reviewer comment 2: external-validity scope audit

No additional phrase in the manuscript should call the two BDG aggregates independent sites or frozen transfer. This phase creates an auditable claim table and, when `archive.zip` is available, reports metadata-only university-site availability without using model performance to select a site.

In [ ]:
# 15. External-scope audit and optional BDG metadata inventory.
import zipfile

external_scope = pd.DataFrame([
    {
        "dataset": "BDG_EDU",
        "study_role": "public external-source check",
        "same_BDG_site_as_other_external_aggregate": True,
        "dataset_local_HPO": True,
        "zero_shot_transfer": False,
        "frozen_cross_site_transfer": False,
        "cross_climate_generalization_claim_allowed": False,
        "recommended_manuscript_phrase": "public external-source check with dataset-local development",
    },
    {
        "dataset": "BDG_DORM",
        "study_role": "public external-source check",
        "same_BDG_site_as_other_external_aggregate": True,
        "dataset_local_HPO": True,
        "zero_shot_transfer": False,
        "frozen_cross_site_transfer": False,
        "cross_climate_generalization_claim_allowed": False,
        "recommended_manuscript_phrase": "public external-source check with dataset-local development",
    },
])
external_scope.to_csv(OUTPUTS["claim_audit"] / "BDG_external_validity_claim_audit.csv", index=False)

archive_candidates = [PROJECT_ROOT / "archive.zip", DATA_DIR / "archive.zip"]
archive_path = next((p for p in archive_candidates if p.exists()), None)
site_inventory = pd.DataFrame()
if archive_path is not None:
    with zipfile.ZipFile(archive_path) as zf:
        names = set(zf.namelist())
        if "meta_open.csv" in names:
            meta = pd.read_csv(zf.open("meta_open.csv"))
            university = meta.loc[meta["subindustry"].eq("College/University")].copy()
            if {"newweatherfilename", "timezone", "annualschedule"}.issubset(university.columns):
                site_inventory = (
                    university.groupby(["newweatherfilename", "timezone", "annualschedule"], dropna=False)
                    .agg(
                        n_buildings=("primaryspaceuse_abbrev", "size"),
                        n_use_codes=("primaryspaceuse_abbrev", "nunique"),
                    )
                    .reset_index()
                    .sort_values("n_buildings", ascending=False)
                )
                site_inventory["weather_file_available"] = site_inventory["newweatherfilename"].isin(names)
                site_inventory["schedule_file_available"] = site_inventory["annualschedule"].isin(names)
                site_inventory.to_csv(OUTPUTS["claim_audit"] / "BDG_university_site_metadata_inventory.csv", index=False)
else:
    print("archive.zip not found; claim audit is still complete. Site inventory skipped.")

display(external_scope)
if not site_inventory.empty:
    display(site_inventory.head(20))

## Phase IV — Reviewer comment 5: common constrained downstream scheduling sensitivity

This is **not** presented as realized HVAC, tariff, comfort, or financial performance. It is a common storage-dispatch scenario applied identically to LASH, the selected reduced architecture, and the focal comparator. Battery capacity and power are scaled using the **Train+Validation mean load**, never the test target. Each schedule is optimized from the model forecast and evaluated against the subsequently observed load. An oracle using actual load is reported only as an upper-bound reference.

In [ ]:
# 16. Constrained battery peak-shaving LP.
from scipy.optimize import linprog


def optimize_storage_peak_shaving(
    predicted_load,
    *,
    e_max,
    p_max,
    eta_roundtrip=0.90,
    soc_fraction_initial=0.50,
    throughput_penalty=1e-5,
):
    y = np.asarray(predicted_load, dtype=float)
    H = len(y)
    eta_c = math.sqrt(eta_roundtrip)
    eta_d = math.sqrt(eta_roundtrip)
    soc0 = soc_fraction_initial * e_max

    # variables: charge[H], discharge[H], soc[H+1], peak
    i_c = 0
    i_d = H
    i_s = 2 * H
    i_peak = 3 * H + 1
    nvar = i_peak + 1

    c = np.zeros(nvar)
    c[i_c:i_c+H] = throughput_penalty
    c[i_d:i_d+H] = throughput_penalty
    c[i_peak] = 1.0

    A_eq, b_eq = [], []
    row = np.zeros(nvar); row[i_s] = 1.0
    A_eq.append(row); b_eq.append(soc0)
    for h in range(H):
        row = np.zeros(nvar)
        row[i_s+h+1] = 1.0
        row[i_s+h] = -1.0
        row[i_c+h] = -eta_c
        row[i_d+h] = 1.0 / eta_d
        A_eq.append(row); b_eq.append(0.0)
    # terminal SOC equals initial SOC: no hidden end-of-horizon energy borrowing.
    row = np.zeros(nvar); row[i_s+H] = 1.0
    A_eq.append(row); b_eq.append(soc0)

    A_ub, b_ub = [], []
    for h in range(H):
        # load + charge - discharge <= peak
        row = np.zeros(nvar)
        row[i_c+h] = 1.0
        row[i_d+h] = -1.0
        row[i_peak] = -1.0
        A_ub.append(row); b_ub.append(-y[h])

    bounds = (
        [(0.0, p_max)] * H
        + [(0.0, p_max)] * H
        + [(0.0, e_max)] * (H + 1)
        + [(0.0, None)]
    )
    res = linprog(
        c,
        A_ub=np.asarray(A_ub), b_ub=np.asarray(b_ub),
        A_eq=np.asarray(A_eq), b_eq=np.asarray(b_eq),
        bounds=bounds,
        method="highs",
    )
    if not res.success:
        raise RuntimeError(res.message)
    x = res.x
    charge = x[i_c:i_c+H]
    discharge = x[i_d:i_d+H]
    soc = x[i_s:i_s+H+1]
    return {
        "charge": charge,
        "discharge": discharge,
        "soc": soc,
        "predicted_peak": float(x[i_peak]),
        "e_max": float(e_max),
        "p_max": float(p_max),
    }


def realized_schedule_metrics(actual_load, schedule):
    actual = np.asarray(actual_load, float)
    grid = actual + schedule["charge"] - schedule["discharge"]
    original_peak = float(actual.max())
    realized_peak = float(grid.max())
    reduction = original_peak - realized_peak
    return {
        "original_peak": original_peak,
        "realized_peak": realized_peak,
        "peak_reduction": reduction,
        "peak_reduction_pct": 100.0 * reduction / original_peak if original_peak > 0 else np.nan,
        "throughput": float(np.sum(schedule["charge"] + schedule["discharge"])),
        "soc_terminal_error": float(abs(schedule["soc"][-1] - schedule["soc"][0])),
    }


def seed_mean_prediction(payloads):
    return np.mean(np.stack([np.asarray(p["prediction"], float) for p in payloads]), axis=0)

In [ ]:
# 17. Run the common storage-dispatch sensitivity on non-overlapping 24-hour trajectories.
storage_rows = []

for dataset_key in ALL_DATASETS:
    # Build pretest mean load only for battery sizing; test target is never used for sizing.
    bundles = build_full_bundles(dataset_key)
    pretest = core.concat_bundles(bundles["train"], bundles["val"], split="train_plus_validation")
    pretest_mean_load = float(np.mean(pretest.y))

    # Use the full-LASH actual array as the common target reference.
    full_payloads = [load_full_pred(dataset_key, s) for s in CONFIRMATORY_SEEDS]
    actual = np.asarray(full_payloads[0]["actual"], float)
    full_mean_pred = seed_mean_prediction(full_payloads)
    reduced_payloads = [load_reduced_pred(dataset_key, s) for s in CONFIRMATORY_SEEDS]
    reduced_mean_pred = seed_mean_prediction(reduced_payloads)

    model_predictions = {
        "LASH": full_mean_pred,
        "LASH_REDUCED": reduced_mean_pred,
    }
    for comparator in FOCAL_COMPARATORS[dataset_key]:
        if comparator == "LIGHTGBM":
            cps = [load_prediction(_prediction_file(OUTPUTS["seed_extension"], dataset_key, comparator, s)) for s in CONFIRMATORY_SEEDS]
            model_predictions[comparator] = seed_mean_prediction(cps)
        else:
            cp = load_prediction(_prediction_file(OUTPUTS["seed_extension"], dataset_key, comparator, 0))
            model_predictions[comparator] = np.asarray(cp["prediction"], float)

    # Non-overlapping daily trajectories avoid counting the same realized hour in many schedules.
    origin_idx = np.arange(0, actual.shape[0], 24, dtype=int)

    for scenario_name, sc in STORAGE_SCENARIOS.items():
        e_max = sc["duration_h"] * pretest_mean_load
        p_max = sc["power_fraction"] * pretest_mean_load

        # Oracle upper bound for each trajectory, used only after the actual load is observed.
        oracle_metrics = {}
        for idx in origin_idx:
            oracle_schedule = optimize_storage_peak_shaving(
                actual[idx], e_max=e_max, p_max=p_max
            )
            oracle_metrics[idx] = realized_schedule_metrics(actual[idx], oracle_schedule)

        for model_name, pred in model_predictions.items():
            daily = []
            for idx in origin_idx:
                schedule = optimize_storage_peak_shaving(
                    pred[idx], e_max=e_max, p_max=p_max
                )
                m = realized_schedule_metrics(actual[idx], schedule)
                oracle_red = oracle_metrics[idx]["peak_reduction"]
                m["oracle_capture"] = (
                    m["peak_reduction"] / oracle_red if oracle_red > 1e-12 else np.nan
                )
                daily.append(m)
            df = pd.DataFrame(daily)
            storage_rows.append({
                "dataset": dataset_key,
                "model": model_name,
                "scenario": scenario_name,
                "n_nonoverlap_24h_schedules": len(df),
                "sizing_reference": "Train+Validation mean load",
                "battery_duration_h_at_pretest_mean": sc["duration_h"],
                "power_fraction_of_pretest_mean": sc["power_fraction"],
                "eta_roundtrip": 0.90,
                "terminal_soc_equal_initial": True,
                "realized_peak_reduction_pct_mean": float(df["peak_reduction_pct"].mean()),
                "realized_peak_reduction_pct_sd": float(df["peak_reduction_pct"].std(ddof=1)),
                "oracle_capture_mean": float(df["oracle_capture"].mean(skipna=True)),
                "throughput_mean": float(df["throughput"].mean()),
                "max_terminal_soc_error": float(df["soc_terminal_error"].max()),
                "interpretation": "scenario-based constrained storage scheduling; not realized BEMS savings",
            })

storage_summary = pd.DataFrame(storage_rows)
storage_summary.to_csv(OUTPUTS["scheduling"] / "common_constrained_storage_scheduling_summary.csv", index=False)
display(storage_summary)

In [ ]:
# 18. Produce a reviewer-evidence map and restrained response/manuscript language based on actual outputs.

def fmt(x, digits=4):
    return "NA" if pd.isna(x) else f"{float(x):.{digits}f}"

primary = pd.read_csv(OUTPUTS["effects"] / "PRIMARY_168h_effect_table.csv")
archsel = json.loads((OUTPUTS["architecture"] / "SELECTED_REDUCED_ARCHITECTURE_LOCKED_BEFORE_TEST.json").read_text(encoding="utf-8"))
rvf = pd.read_csv(OUTPUTS["effects"] / "reduced_vs_full_noninferiority.csv")
compfx_path = OUTPUTS["effects"] / "retained_component_incremental_value.csv"
compfx = pd.read_csv(compfx_path) if compfx_path.exists() else pd.DataFrame()

map_rows = [
    {
        "reviewer_comment": 1,
        "response_strategy": "Narrow superiority claim; 10-seed focal refits; hierarchical dependence-aware inference.",
        "evidence": "02_ten_seed_confirmatory; 04_effect_sizes_and_inference/PRIMARY_168h_effect_table.csv",
        "claim_ceiling": "Numerical ranking is not called superiority when corrected CI includes zero.",
    },
    {
        "reviewer_comment": 2,
        "response_strategy": "Explicitly reclassify BDG as same-site external-source checks with dataset-local development.",
        "evidence": "01_claim_and_external_scope/BDG_external_validity_claim_audit.csv",
        "claim_ceiling": "No cross-site, zero-shot, frozen-transfer, or cross-climate claim.",
    },
    {
        "reviewer_comment": 3,
        "response_strategy": "Validation-only nested reduced-architecture selection, lock before test, then 10-seed final test and drop-one retained-component tests.",
        "evidence": "03_reduced_architecture; 04_effect_sizes_and_inference/retained_component_incremental_value.csv",
        "claim_ceiling": "Independent component necessity claimed only where drop-one evidence supports it.",
    },
    {
        "reviewer_comment": 4,
        "response_strategy": "10 seeds, relative effects, 95% hierarchical CI, block effect size, win rate, Holm correction, 1% revision-stage SESOI.",
        "evidence": "04_effect_sizes_and_inference",
        "claim_ceiling": "Statistical support is separated from practical importance.",
    },
    {
        "reviewer_comment": 5,
        "response_strategy": "Common constrained battery-dispatch scenario with equipment/SOC constraints; operational wording remains scenario-based.",
        "evidence": "05_constrained_storage_scheduling/common_constrained_storage_scheduling_summary.csv",
        "claim_ceiling": "No realized tariff, HVAC, comfort, rebound, or financial-savings claim.",
    },
]
pd.DataFrame(map_rows).to_csv(OUTPUTS["response"] / "reviewer3_five_comment_evidence_map.csv", index=False)

# Build factual response paragraphs; no claim is upgraded beyond the computed evidence.
lines = []
lines.append("# Reviewer 3 — Second-Round Response Draft")
lines.append("")
lines.append("I revised the manuscript to separate numerical ranking, inferential support, practical magnitude, external validity, architectural necessity, and downstream operational sensitivity. Claims were narrowed wherever the available evidence does not justify a stronger interpretation.")
lines.append("")

# Comment 1
lines.append("## Comment 1")
lines.append("I agree that numerical ranking alone should not be described as superiority. I therefore extended the focal stochastic comparisons to ten fixed final-refit seeds while keeping the original local-HPO settings and routing decisions frozen, and I report dependence-aware confidence intervals and multiplicity-adjusted p-values. The manuscript now uses 'numerically ranked first' only as a descriptive statement and reserves 'supported improvement' for contrasts whose corrected interval excludes zero.")
for _, r in primary.iterrows():
    lines.append(
        f"- {r['dataset']} vs {r['comparator']}: ΔNMAE={fmt(r['delta_NMAE_pp_LASH_minus_comparator'])} pp; "
        f"relative improvement={fmt(r['relative_improvement_pct'],3)}%; 95% hierarchical CI for Δ=[{fmt(r['CI_low_delta_pp'])}, {fmt(r['CI_high_delta_pp'])}]; "
        f"Holm p={fmt(r['p_hier_holm'],3)}; interpretation: {r['interpretation']}."
    )
lines.append("")

# Comment 2
lines.append("## Comment 2")
lines.append("I agree that the two BDG aggregates do not constitute cross-site or frozen-transfer validation. Both are now described consistently as public external-source checks from the same anonymized university site, each with dataset-local development. The Ridge-only routing outcomes are interpreted only as validation-gated rejection of unsupported nonlinear correction within those local experiments; they are not used as evidence of cross-site or zero-shot generalization.")
lines.append("")

# Comment 3
lines.append("## Comment 3")
lines.append(
    f"I addressed the architectural concern with a second-round reduced-model protocol that was frozen before additional refits. "
    f"Five nested reduced candidates were developed using Cluster 1 and Cluster 2 training/validation data only, and the common architecture was locked before test evaluation. The selected architecture was **{archsel['selected_architecture']}**. I then refit that architecture on all fmy datasets using ten final seeds and performed drop-one tests for every optional component retained by the selected model."
)
if not compfx.empty:
    for component, g in compfx.groupby("retained_component"):
        supported = int(g["supported_after_Holm"].sum())
        lines.append(f"- {component}: supported after Holm correction in {supported}/{len(g)} dataset-specific tests; the manuscript should avoid claiming universal necessity if support is mixed.")
lines.append("")

# Comment 4
lines.append("## Comment 4")
lines.append(
    f"I agree that the magnitude of improvement must be reported separately from statistical significance. The revision now reports ten-seed mean±SD, absolute and relative effects, hierarchical 95% confidence intervals, weekly-block standardized effects, block win rates, and Holm-adjusted p-values. Before the new refits I fixed a second-round SESOI of {SESOI_REL_NMAE_PCT:.1f}% relative NMAE improvement. This is explicitly described as a reviewer-requested revision-stage interpretation threshold, not as an original preregistration or a universal building-energy threshold. Effects below this threshold are described as modest even when statistically supported."
)
lines.append("")

# Comment 5
lines.append("## Comment 5")
lines.append("I agree that the original top-k curtailment proxy is insufficient for a control-level operational claim. I therefore restrict the wording to offline/scenario-based sensitivity and add a common constrained battery-dispatch experiment. Every forecast is passed to the same linear storage scheduler with explicit power, energy, efficiency, SOC, and terminal-SOC constraints; battery size is normalized using Train+Validation mean load rather than test demand. Realized peak reduction is evaluated against the subsequently observed load, and an oracle is reported only as an upper-bound reference. I continue to make no claim of realized tariff savings, HVAC efficiency, comfort improvement, rebound behavior, or production BEMS performance.")

response_path = OUTPUTS["response"] / "reviewer3_second_round_response_draft.md"
response_path.write_text("\n".join(lines), encoding="utf-8")

manuscript = f"""# Manuscript wording guardrails generated by the second-round notebook

## Performance claim
Use: **competitive forecasting performance with statistically supported improvements where explicitly demonstrated**.
Do not use: **universally superior**, **outperforms all baselines**, or equivalent wording.

## External validity
Use: **BDG_Edu and BDG_Dorm are public external-source checks with dataset-local development from the same anonymized site**.
Do not use: **independent cross-site validation**, **zero-shot transfer**, **frozen transfer**, or **cross-climate validation**.

## Reduced architecture
The validation-locked reduced candidate is: **{SELECTED_REDUCED_ARCH}**.
Report the full-vs-reduced 10-seed result and the retained-component drop-one table. If any retained component has mixed evidence, describe it as retained by the validation-selected architecture rather than universally necessary.

## Practical significance
Separate p-values from magnitude. The second-round SESOI is **{SESOI_REL_NMAE_PCT:.1f}% relative NMAE**, fixed before the additional refits and not claimed as an original preregistration.

## Operations
Use: **constrained storage-dispatch sensitivity** / **offline downstream scheduling scenario**.
Do not claim realized tariff savings, HVAC control benefit, comfort improvement, rebound handling, or production readiness.
"""
(OUTPUTS["response"] / "manuscript_wording_guardrails.md").write_text(manuscript, encoding="utf-8")

print("Response draft:", response_path)
print("Manuscript guardrails:", OUTPUTS["response"] / "manuscript_wording_guardrails.md")

In [ ]:
# 19. Final verification, inventory, and completion marker.
required_outputs = [
    OUTPUTS["protocol"] / "SECOND_REVIEW_PLAN_FROZEN_BEFORE_REFITS.json",
    OUTPUTS["seed_extension"] / "all_focal_seed_summary.csv",
    OUTPUTS["effects"] / "PRIMARY_168h_effect_table.csv",
    OUTPUTS["architecture"] / "SELECTED_REDUCED_ARCHITECTURE_LOCKED_BEFORE_TEST.json",
    OUTPUTS["effects"] / "reduced_vs_full_noninferiority.csv",
    OUTPUTS["claim_audit"] / "BDG_external_validity_claim_audit.csv",
    OUTPUTS["scheduling"] / "common_constrained_storage_scheduling_summary.csv",
    OUTPUTS["response"] / "reviewer3_second_round_response_draft.md",
]
missing = [str(p) for p in required_outputs if not p.exists()]
if missing:
    raise RuntimeError("Second-review completion check failed; missing:\n" + "\n".join(missing))

inventory = []
for p in SECOND_REVIEW_DIR.rglob("*"):
    if p.is_file():
        inventory.append({
            "relative_path": str(p.relative_to(SECOND_REVIEW_DIR)),
            "bytes": p.stat().st_size,
            "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
        })
pd.DataFrame(inventory).sort_values("relative_path").to_csv(
    SECOND_REVIEW_DIR / "SECOND_REVIEW_OUTPUT_INVENTORY_SHA256.csv", index=False
)

completion = {
    "status": "completed",
    "protocol": PROTOCOL_NAME,
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "wall_hours_current_kernel": (time.perf_counter() - RUN_STARTED_PERF) / 3600.0,
    "original_results_modified": False,
    "second_review_plan_sha256": plan_sha,
    "selected_reduced_architecture": SELECTED_REDUCED_ARCH,
    "confirmatory_seed_count": len(CONFIRMATORY_SEEDS),
    "primary_dependence_block_h": PRIMARY_BLOCK_HOURS,
    "revision_stage_SESOI_relative_NMAE_pct": SESOI_REL_NMAE_PCT,
    "external_transfer_claim": False,
    "downstream_scheduling_scope": "constrained storage scenario; not realized BEMS benefit",
    "hardware": hardware,
}
completion_path = SECOND_REVIEW_DIR / "SECOND_REVIEW_COMPLETED_20260829.json"
completion_path.write_text(json.dumps(completion, indent=2, ensure_ascii=False), encoding="utf-8")

print("=" * 84)
print("SECOND-ROUND REVIEWER PIPELINE: COMPLETE")
print("=" * 84)
print(json.dumps(completion, indent=2, ensure_ascii=False))
print("All outputs:", SECOND_REVIEW_DIR)

## Interpretation rule after the run

Do **not** rewrite the paper from numerical rank alone. Use the generated `PRIMARY_168h_effect_table.csv`, `reduced_vs_full_noninferiority.csv`, `retained_component_incremental_value.csv`, and `common_constrained_storage_scheduling_summary.csv` together.

The safest second-round narrative is deliberately moderate: LASH is a leakage-audited validation-gated forecasting protocol; statistically supported gains are stated only where corrected uncertainty supports them; small effects are called modest; the BDG experiments are not transfer validation; the reduced architecture is chosen on validation data before test; and downstream conclusions remain scenario-based even after adding explicit storage constraints.